<a href="https://colab.research.google.com/github/SNK0-0/Bit-Net-LLM_Proj/blob/main/BitNet_LLM_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ------------------------------------------------------------------------------
# STARTUP CHECKLIST FOR EVERY NEW SESSION (Your 2-minute setup routine)
# ------------------------------------------------------------------------------
# [ ] SET RUNTIME: Go to Runtime -> Change runtime type.
#     -> Choose CPU for writing code and debugging layer architectures.
#     -> Choose T4 GPU only when ready to initiate an active training execution cell.
# [ ] LAUNCH ENVIRONMENT: Run Cell 1 (Mounts Drive, clones fresh repo, installs pip, builds tokens).
# [ ] REGENERATE CUSTOM CODE: Run all your `%%writefile` cells to inject modules into the VM.
# [ ] ROUTE & RUN: Fire off your training loop with output paths pointing directly to your Drive.
# ==============================================================================

In [5]:
# 1. Mount your permanent storage
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone the repository into the volatile VM storage
!git clone https://github.com/karpathy/nanoGPT.git
%cd nanoGPT

# 3. Install required libraries
!pip install torch numpy transformers datasets tiktoken wandb tqdm

# 4. Prepare the dataset tokens
!python data/shakespeare_char/prepare.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'nanoGPT'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 981.25 KiB | 8.84 MiB/s, done.
Resolving deltas: 100% (380/380), done.
/content/nanoGPT/nanoGPT
length of dataset in characters: 1,115,394
all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens


In [ ]:
!python train.py config/train_shakespeare_char.py --out_dir='/content/drive/MyDrive/BitNet_Project/baseline' --max_iters=1500

Overriding config with config/train_shakespeare_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of 

In [ ]:
# Final train loss : 1.1510
# Final validation loss : 1.4739
# avg. iteration time : 570-580 ms
# Model flops utilization : 0.63%-0.64%

In [6]:
%%writefile bitlinear.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class BitLinear(nn.Linear):
    """
    Custom Linear layer for BitNet b1.58.
    Quantizes weights to ternary (-1, 0, 1) and activations to 8-bit (int8).
    """
    def __init__(self, in_features, out_features, bias=False, eps=1e-5):
        # We inherit from nn.Linear so self.weight and self.bias are handled automatically
        super().__init__(in_features, out_features, bias)
        self.eps = eps

    def forward(self, x):
        # -------------------------------------------------------------
        # 1. Weight Quantization (Ternary)
        # -------------------------------------------------------------
        w = self.weight

        # Calculate gamma: the mean absolute value of the weight matrix
        gamma = w.abs().mean()

        # Scale, round, and clamp to ternary range [-1, 0, 1]
        w_scaled = w / (gamma + self.eps)
        w_quant = torch.clamp(torch.round(w_scaled), -1, 1)

        # Straight-Through Estimator (STE) trick for weights
        # Forward pass uses w_quant, backward pass updates original w
        w_quant = w + (w_quant - w).detach()

        # -------------------------------------------------------------
        # 2. Activation Quantization (8-bit)
        # -------------------------------------------------------------
        # Calculate beta: the max absolute value per token/feature
        beta = x.abs().max(dim=-1, keepdim=True).values

        # Scale to 8-bit integer range [-128, 127], round, and clamp
        x_scaled = x * (127.0 / (beta + self.eps))
        x_quant = torch.clamp(torch.round(x_scaled), -128, 127)

        # Straight-Through Estimator (STE) trick for activations
        x_quant = x + (x_quant - x).detach()

        # -------------------------------------------------------------
        # 3. The Core Matrix Multiplication
        # -------------------------------------------------------------
        # F.linear performs: x_quant @ w_quant^T + bias
        output = F.linear(x_quant, w_quant, self.bias)

        # -------------------------------------------------------------
        # 4. Dequantization / Rescaling
        # -------------------------------------------------------------
        # We divided w by gamma and x by (beta/127), so we must multiply them back
        scale_factor = (gamma * beta) / 127.0
        output = output * scale_factor

        return output

Writing bitlinear.py


In [7]:
# Create a new cell and run this to generate model_bitnet.py
import os

# Read the original nanoGPT model architecture
with open('model.py', 'r') as f:
    model_code = f.read()

# Inject our custom module import
bitnet_imports = """
import math
import inspect
from bitlinear import BitLinear
"""

# Define the targeted Attention layer replacements
old_attention = """        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)"""

new_attention = """        # key, query, value projections for all heads using BitLinear
        self.c_attn = BitLinear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection using BitLinear
        self.c_proj = BitLinear(config.n_embd, config.n_embd, bias=config.bias)"""

# Define the targeted MLP layer replacements
old_mlp = """        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)"""

new_mlp = """        self.c_fc    = BitLinear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = BitLinear(4 * config.n_embd, config.n_embd, bias=config.bias)"""

# Apply transformations and write to disk
modified_code = model_code.replace("import math\nimport inspect", bitnet_imports)
modified_code = modified_code.replace(old_attention, new_attention)
modified_code = modified_code.replace(old_mlp, new_mlp)

with open('model_bitnet.py', 'w') as f:
    f.write(modified_code)

print("model_bitnet.py successfully generated with BitLinear modules injected.")

model_bitnet.py successfully generated with BitLinear modules injected.


In [8]:
# Create a new cell and run this to generate train_bitnet.py
import os

# Read the original nanoGPT training script
with open('train.py', 'r') as f:
    train_script = f.read()

# Reroute the import architecture
modified_train = train_script.replace(
    "from model import GPTConfig, GPT",
    "from model_bitnet import GPTConfig, GPT"
)

# Write out the new executable engine
with open('train_bitnet.py', 'w') as f:
    f.write(modified_train)

print("train_bitnet.py engine complete and ready to deploy.")

train_bitnet.py engine complete and ready to deploy.


In [ ]:
!python train_bitnet.py config/train_shakespeare_char.py --out_dir='/content/drive/MyDrive/BitNet_project/quantized_run' --max_iters=1500

Overriding config with config/train_shakespeare_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of 

In [ ]:
# ==============================================================================
# PHASE 5 RESULTS: 1.58-BIT QUANTIZED RUN (DEV B TRACK)
# ==============================================================================
# Final Train Loss (Step 1500): 2.4712
# Final Val Loss   (Step 1500): 2.4870
# Model MFU                   : 0.47%
#
# NOTES:
# - Checkpoint successfully routed to: /content/drive/MyDrive/bitnet_project/quantized_run
# - Val loss is higher than FP16 baseline (expected without hyperparameter tuning).
# - MFU is lower because the custom BitLinear layer relies on unoptimized Python ops
#   rather than custom CUDA kernels. Mathematical logic is verified and functioning.
# ==============================================================================

In [15]:
%%writefile generate.py
import torch
import pickle
import os
from model_bitnet import GPT, GPTConfig

# 1. Hardware and Path Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Ensure this matches your exact Drive path capitalization
out_dir = '/content/drive/MyDrive/BitNet_Project/quantized_run'
ckpt_path = os.path.join(out_dir, 'ckpt.pt')

if not os.path.exists(ckpt_path):
    print(f"❌ Error: Checkpoint not found at {ckpt_path}.")
else:
    print(f"Loading BitNet b1.58 model from {ckpt_path}...")
    checkpoint = torch.load(ckpt_path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)

    # -------------------------------------------------------------
    # 2. THE FIX: Strip the '_orig_mod.' prefix from compiled weights
    # -------------------------------------------------------------
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'

    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            # Replace the old key with the new, stripped key
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    # Now load the cleaned dictionary into the model
    model.load_state_dict(state_dict)
    # -------------------------------------------------------------

    model.eval()
    model.to(device)

    # 3. Load the Shakespeare Character Vocabulary
    meta_path = os.path.join('data', 'shakespeare_char', 'meta.pkl')
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    stoi, itos = meta['stoi'], meta['itos']

    # 4. Set the Prompt
    prompt = "O Romeo"
    x = torch.tensor([stoi[c] for c in prompt], dtype=torch.long, device=device).unsqueeze(0)

    # 5. Generate Text!
    print("\n--- Generating Text from 1.58-Bit Model ---\n")
    with torch.no_grad():
        y = model.generate(x, max_new_tokens=500, temperature=0.8)
        generated_text = ''.join([itos[int(i)] for i in y[0]])
        print(generated_text)

Overwriting generate.py


In [16]:
!python generate.py

Loading BitNet b1.58 model from /content/drive/MyDrive/BitNet_Project/quantized_run/ckpt.pt...
number of parameters: 10.65M

--- Generating Text from 1.58-Bit Model ---

O Romeo har t baturong IUTonortore ather s sind st sinem  hitintouvit basimy the An bens ar thad
KAn agithedotor of ONTha h me kes t merd herosond I achaund y her t fusth chis the th, ath my byow wisais teand d surereror po tofou, d arinddsull.
AUCEThathowinowe ksthy ingurond C:
Wh this as t t fy HELAnth ar stst,
Henlad, w sp s my
Thent arde kerererer herd thinoun, med f henil INube.
S:

Faundisincher aredute:
G th helld mendr woowiserimagr d S:

I wabeime trd fanervere I'suldowiratou mars thathed stha
